# 🎧 Compréhension Audio — Qwen2-Audio-7B-Instruct (Google Colab)

Run the FastAPI backend **and** the web UI on a Colab **A100** GPU, then open them through a public URL.

**Before you start:** `Runtime → Change runtime type → Hardware accelerator → A100 GPU`.

Then run the cells **top to bottom**. The first run downloads ~16 GB of weights, so give it a few minutes.

## 1 · Check the GPU

In [ ]:
!nvidia-smi

## 2 · Clone the project from GitHub

Pulls the repo into `/content/Tasnim`.

In [ ]:
import os

REPO_URL = 'https://github.com/mouadelhaddad/Tasnim.git'
CLONE_DIR = '/content/Tasnim'

# Re-clone fresh each time so you always run the latest code.
!rm -rf {CLONE_DIR}
!git clone {REPO_URL} {CLONE_DIR}

BACKEND_DIR = os.path.join(CLONE_DIR, 'backend')
PROJECT_ROOT = CLONE_DIR
assert os.path.isfile(os.path.join(BACKEND_DIR, 'app', 'main.py')), \
    'backend/app/main.py not found — check the repo layout.'
print('BACKEND_DIR  =', BACKEND_DIR)

In [ ]:
# Alternative — upload a ZIP instead of cloning (uncomment to use):
# from google.colab import files
# import zipfile, glob, os
# uploaded = files.upload()
# for name in uploaded:
#     if name.endswith('.zip'):
#         zipfile.ZipFile(name).extractall('/content/project')
# m = glob.glob('/content/**/backend/app/main.py', recursive=True)
# BACKEND_DIR = os.path.dirname(os.path.dirname(m[0]))
# PROJECT_ROOT = os.path.dirname(BACKEND_DIR)
# print('BACKEND_DIR =', BACKEND_DIR)

## 3 · Install dependencies

Colab already ships PyTorch + CUDA, so we **don't** reinstall torch (that can break the GPU build). We only add the model, audio, and API libraries.

In [ ]:
!apt-get -qq install -y ffmpeg
!pip install -q "transformers>=4.45.0" accelerate \
    librosa==0.10.2 soundfile==0.12.1 \
    fastapi==0.115.5 "uvicorn[standard]==0.32.1" python-multipart==0.0.12 \
    "pydantic>=2.0.0" python-dotenv==1.0.1
print('Dependencies installed.')

## 4 · (Optional) HuggingFace token

Qwen2-Audio is public, so a token usually isn't required. Uncomment only if you hit download rate limits.

In [ ]:
import os
# from getpass import getpass
# os.environ['HF_TOKEN'] = getpass('HF token: ')

## 5 · Configure the model

A100 has plenty of VRAM, so we load the full **fp16** model — no quantization, no mock.

In [ ]:
import os, torch

name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (no GPU!)'
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU: {name}  ({total_gb:.1f} GB)')

os.environ['MODEL_NAME'] = 'Qwen/Qwen2-Audio-7B-Instruct'
os.environ['USE_MOCK'] = 'false'      # real model
os.environ['LOAD_IN_4BIT'] = 'false'  # full fp16 on A100
print('-> Full fp16, real model.')

## 6 · Download the public-tunnel tool (cloudflared)

This exposes the local server on a temporary `*.trycloudflare.com` URL — no signup needed.

In [ ]:
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
print('cloudflared ready.')

## 7 · Launch the server + tunnel

Starts uvicorn (which loads the model on startup) and opens the public URL.

In [ ]:
import subprocess, time, re, os

UVICORN_LOG = '/content/uvicorn.log'
TUNNEL_LOG  = '/content/cloudflared.log'

# FastAPI server — loads the model on startup (first run downloads the weights).
server = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND_DIR,
    stdout=open(UVICORN_LOG, 'w'), stderr=subprocess.STDOUT,
    env={**os.environ},
)
print('uvicorn started (PID %d)' % server.pid)

# Public tunnel.
tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=open(TUNNEL_LOG, 'w'), stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(40):
    time.sleep(2)
    try:
        log = open(TUNNEL_LOG).read()
    except FileNotFoundError:
        continue
    m = re.search(r'https://[-\w.]+\.trycloudflare\.com', log)
    if m:
        public_url = m.group(0)
        break

print('\n' + '=' * 64)
if public_url:
    print('🌐  Your app will be available here:\n     ' + public_url)
    print('\n    (The page works once the model has finished loading — see step 8.)')
else:
    print('Could not detect the tunnel URL. Check: !cat /content/cloudflared.log')
print('=' * 64)

## 8 · Wait for the model to finish loading

Polls the health endpoint. The first run downloads ~16 GB, so this can take several minutes.

In [ ]:
import time, requests

print('Loading model', end='', flush=True)
ready = False
for _ in range(180):  # up to ~30 min
    try:
        r = requests.get('http://localhost:8000/api/v1/health', timeout=5)
        if r.ok and r.json().get('model_loaded'):
            print('\n\n✅ Ready:', r.json())
            print('\n🌐 Open:', public_url)
            ready = True
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(10)

if not ready:
    print('\nStill not ready — inspect the log below.')
    print(open('/content/uvicorn.log').read()[-2000:])

## 9 · Logs & troubleshooting

- **Server logs:** run the cell below.
- **Tunnel URL missing?** `!cat /content/cloudflared.log`
- **Keep this tab open** — closing Colab or hitting the idle timeout stops the server and invalidates the URL.
- The `trycloudflare.com` URL is **temporary** and changes every launch.
- Pushed new code? Re-run **step 2** (it re-clones) then steps 7 → 8.

In [ ]:
!tail -n 60 /content/uvicorn.log